# 03 — Failure Analysis

This notebook explores misclassified examples after running evaluation.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

results_path = Path('../outputs/evaluation_results/evaluation_results.csv')
if not results_path.exists():
    print("Run: python -m evaluation.evaluate  first to generate results.")
else:
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} evaluation results")
    df.head()


## Overall Accuracy

In [ ]:
if results_path.exists():
    acc = (df['true_intent'] == df['predicted_intent']).mean()
    print(f"Overall Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    failures = df[df['true_intent'] != df['predicted_intent']]
    print(f"Failures: {len(failures)} / {len(df)} ({len(failures)/len(df)*100:.1f}%)")


## Failure Cases — Low Confidence First

In [ ]:
if results_path.exists():
    failures_sorted = failures.sort_values('confidence')
    display_cols = ['message', 'true_intent', 'predicted_intent', 'confidence', 'should_escalate']
    print(failures_sorted[display_cols].head(10).to_string(index=False))


## Confusion Matrix

In [ ]:
if results_path.exists():
    from sklearn.metrics import confusion_matrix
    from src.config import INTENTS
    cm = confusion_matrix(df['true_intent'], df['predicted_intent'], labels=INTENTS)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=INTENTS, yticklabels=INTENTS, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix (Counts)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Confidence vs Correctness

In [ ]:
if results_path.exists():
    df['correct'] = df['true_intent'] == df['predicted_intent']
    print("Mean confidence by correctness:")
    print(df.groupby('correct')['confidence'].describe().round(4))


## Common Failure Patterns

In [ ]:
if results_path.exists():
    failure_pairs = failures.groupby(['true_intent', 'predicted_intent']).size().reset_index(name='count')
    failure_pairs = failure_pairs.sort_values('count', ascending=False)
    print("Most common confusion pairs:")
    print(failure_pairs.head(10).to_string(index=False))


## Conclusions

Common failure patterns:
- Overlapping intents (complaint_angry vs billing_issue)
- Short messages with insufficient context
- Ambiguous phrasing that could belong to multiple intents

The escalation safety net (confidence < 0.60) catches many of these low-confidence predictions.